Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-ollama sentence-transformers neo4j torch requests ragas datasets scikit-learn pandas -q

print("[All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.8/325.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires request

Install and Start Ollama

In [ ]:
# Install Ollama
!curl https://ollama.ai/install.sh | sh -s &>/dev/null
print("[Ollama installed!")



import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print(" Ollama server started!")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  53116      0 --:--:-- --:--:-- --:--:-- 53337
[Ollama installed!
Starting Ollama server...
 Ollama server started!


In [ ]:
!ollama pull mistral
print("Mistral model downloaded!")


Mistral model downloaded!


In [ ]:
import json
from typing import Dict, List, Any
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.schema import Document
from neo4j import GraphDatabase, Driver
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass


 DATA STRUCTURES

In [ ]:
@dataclass
class HybridRAGResult:
    answer: str
    domain_graph_context: str
    vector_context: str
    sources_used: Dict[str, Any]

CUSTOM RETRIEVERS

In [ ]:
class DomainGraphRetriever:
    """Retriever for Neo4j domain graph (simplified, no BaseRetriever)"""

    def __init__(self, neo4j_uri: str, neo4j_user: str, neo4j_password: str, llm):
        """Initialize with proper parameters"""
        self.neo4j_uri = neo4j_uri
        self.neo4j_user = neo4j_user
        self.neo4j_password = neo4j_password
        self.llm = llm
        self.driver = GraphDatabase.driver(
            neo4j_uri,
            auth=(neo4j_user, neo4j_password)
        )

    def _generate_cypher(self, query: str) -> str:
        """Generate Cypher query from natural language"""
        cypher_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a Neo4j Cypher expert. Convert this question to a Cypher query.

Database Schema:
- Species: name, id, morphological_keys, genetic_markers
- Host: id, type, species
- Location: name, id, bioclimatic_zone
- Disease: name, id
- Article: title, source_file, id, publication_year

Relationships: TRANSMITS, PARASITIZES, FOUND_IN, STUDIES, CONDUCTED_IN

RULES - DO NOT USE:
ALL(), ANY(), APOC functions
Complex filters
Aggregations unless necessary
CASE statements

USE ONLY:
Simple MATCH patterns
WHERE with = or CONTAINS
OPTIONAL MATCH for optional relationships
RETURN specific properties
COLLECT for grouping
ORDER BY
LIMIT

Question: {question}

Generate ONLY valid Cypher. Examples:

Q: "Where is Hyalomma marginatum found?"
A: MATCH (s:Species {{name: 'Hyalomma marginatum'}})-[:FOUND_IN]->(l:Location)
RETURN s.name, s.genetic_markers, s.morphological_keys, l.name, l.bioclimatic_zone

Q: "What species transmit Q fever?"
A: MATCH (s:Species)-[:TRANSMITS]->(d:Disease {{name: 'Q fever'}})
RETURN s.name, s.genetic_markers, s.morphological_keys

Q: "What hosts do Hyalomma parasitize?"
A: MATCH (s:Species {{name: 'Hyalomma marginatum'}})-[:PARASITIZES]->(h:Host)
RETURN s.name, h.type, h.species

Q: "Find species in tropical zones"
A: MATCH (s:Species)-[:FOUND_IN]->(l:Location)
WHERE l.bioclimatic_zone CONTAINS 'tropical'
RETURN s.name, s.genetic_markers, l.name, l.bioclimatic_zone

Return ONLY the query, nothing else:"""
)
        chain = cypher_prompt | self.llm
        cypher = chain.invoke({"question": query}).strip()
        if "```" in cypher:
            cypher = cypher.split("```")[1].replace("cypher", "").strip()
        return cypher

    def get_relevant_documents(self, query: str) -> List[Document]:
        """Execute Cypher query and return as Documents"""
        try:
            cypher = self._generate_cypher(query)
            print(f"[Domain Graph] Generated Cypher: {cypher[:80]}...")

            with self.driver.session() as session:
                result = session.run(cypher)
                records = [dict(record) for record in result]

            # Convert records to Documents
            docs = []
            for i, record in enumerate(records):
                content = json.dumps(record, indent=2)
                doc = Document(
                    page_content=content,
                    metadata={
                        "source": "domain_graph",
                        "record_index": i,
                        "cypher_query": cypher
                    }
                )
                docs.append(doc)

            if not docs:
                doc = Document(
                    page_content="No structured data found in domain graph.",
                    metadata={"source": "domain_graph", "empty": True}
                )
                docs.append(doc)

            return docs
        except Exception as e:
            print(f"[Domain Graph Error] {e}")
            return [Document(
                page_content=f"Error querying domain graph: {str(e)}",
                metadata={"source": "domain_graph", "error": True}
            )]

    def close(self):
        """Close connection"""
        self.driver.close()

In [ ]:
class VectorGraphRetriever:
    """Retriever for vector similarity search"""

    def __init__(self, neo4j_uri: str, neo4j_user: str, neo4j_password: str,
                 embedding_model, top_k: int = 5):
        """Initialize with proper parameters"""
        self.neo4j_uri = neo4j_uri
        self.neo4j_user = neo4j_user
        self.neo4j_password = neo4j_password
        self.embedding_model = embedding_model
        self.top_k = top_k
        self.similarity_threshold = 0.5
        self.driver = GraphDatabase.driver(
            neo4j_uri,
            auth=(neo4j_user, neo4j_password)
        )

    def get_relevant_documents(self, query: str) -> List[Document]:
        """Perform semantic search on vector graph"""
        try:
            # Generate embedding
            embedding = self.embedding_model.encode(query, convert_to_tensor=False).tolist()
            print(f"[Vector Search] Generated embedding of dimension {len(embedding)}")

            # Cypher query for similarity search
            cypher_query = f"""
            MATCH (article:VectorArticle)-[:HAS_VECTOR_CHUNK]->(chunk:VectorChunk)
            WITH chunk, article,
                 gds.similarity.cosine(chunk.embedding, $embedding) AS similarity
            WHERE similarity > {self.similarity_threshold}
            RETURN article.title, chunk.text, chunk.section, similarity, chunk.chunk_id
            ORDER BY similarity DESC
            LIMIT {self.top_k}
            """

            with self.driver.session() as session:
                result = session.run(cypher_query, embedding=embedding)
                records = [dict(record) for record in result]

            # Convert to Documents
            docs = []
            for i, record in enumerate(records):
                content = f"Article: {record.get('article.title', 'Unknown')}\nSection: {record.get('chunk.section', 'General')}\n\n{record.get('chunk.text', '')}"
                doc = Document(
                    page_content=content,
                    metadata={
                        "source": "vector_search",
                        "article": record.get('article.title', 'Unknown'),
                        "section": record.get('chunk.section', 'General'),
                        "similarity": record.get('similarity', 0),
                        "chunk_id": record.get('chunk.chunk_id', ''),
                        "index": i
                    }
                )
                docs.append(doc)

            if not docs:
                doc = Document(
                    page_content="No relevant articles found in vector search.",
                    metadata={"source": "vector_search", "empty": True}
                )
                docs.append(doc)

            return docs
        except Exception as e:
            print(f"[Vector Search Error] {e}")
            return [Document(
                page_content=f"Error in vector search: {str(e)}",
                metadata={"source": "vector_search", "error": True}
            )]

    def close(self):
        """Close connection"""
        self.driver.close()


HYBRID RAG PIPELINE BUILDER

In [ ]:
class HybridRAGPipeline:
    """Builds and manages the hybrid RAG pipeline using LCEL"""

    def __init__(
        self,
        neo4j_uri: str = "neo4j+s://0568253a.databases.neo4j.io",
        neo4j_user: str = "neo4j",
        neo4j_password: str = "sR3IyAZfLN4p5nhMlsR7_bX99816dSDATZ2B4guGCV0",
        llm_model: str = "mistral",
        embedding_model_name: str = "all-MiniLM-L6-v2",
        ollama_base_url: str = "http://localhost:11434"
    ):
        """Initialize retrievers and LLM"""
        print("[Init] Loading OllamaLLM (using langchain_ollama)...")
        self.llm = OllamaLLM(
            model=llm_model,
            base_url=ollama_base_url,
            temperature=0
        )

        print("[Init] Loading Sentence Transformer embeddings...")
        self.embedding_model = SentenceTransformer(embedding_model_name)

        # FIX 2: Initialize retrievers with all required parameters
        print("[Init] Initializing Domain Graph Retriever...")
        self.domain_retriever = DomainGraphRetriever(
            neo4j_uri=neo4j_uri,
            neo4j_user=neo4j_user,
            neo4j_password=neo4j_password,
            llm=self.llm
        )

        print("[Init] Initializing Vector Graph Retriever...")
        self.vector_retriever = VectorGraphRetriever(
            neo4j_uri=neo4j_uri,
            neo4j_user=neo4j_user,
            neo4j_password=neo4j_password,
            embedding_model=self.embedding_model
        )

        self.llm_model = llm_model
        print("[Init] Pipeline ready!\n")

    def _parallel_retrieve(self, question: str):
        """Retrieve from both sources in parallel"""
        with ThreadPoolExecutor(max_workers=2) as executor:
            domain_future = executor.submit(
                self.domain_retriever.get_relevant_documents, question
            )
            vector_future = executor.submit(
                self.vector_retriever.get_relevant_documents, question
            )
            domain_docs = domain_future.result()
            vector_docs = vector_future.result()

        return domain_docs, vector_docs

    def _format_context(self, docs: List[Document], source_name: str) -> str:
        """Format documents into a context string"""
        if not docs:
            return f"No results from {source_name}."

        formatted = []
        for i, doc in enumerate(docs, 1):
            if "similarity" in doc.metadata:
                sim = doc.metadata.get("similarity", 0)
                formatted.append(
                    f"{i}. [{doc.metadata.get('article', '')}] (similarity: {sim:.2f})\n{doc.page_content}"
                )
            else:
                formatted.append(f"{i}. {doc.page_content}")

        return "\n---\n".join(formatted)

    def build_rag_chain(self):
        """
        Build the RAG chain using LCEL
        Format: {retriever1, retriever2, question} | prompt | llm
        """

        # Define prompt template
        prompt = PromptTemplate(
            input_variables=["domain_context", "vector_context", "question"],
            template="""You are an expert in parasitology and vector-borne diseases in Tunisia.

Your goal is to provide an accurate, concise, and evidence-grounded answer.

User Question:
{question}

---

Structured Data from Domain Graph:
{domain_context}

Supporting Context from Vector Articles:
{vector_context}

---

### INSTRUCTIONS:

1. **Domain Graph Facts First**
   - Identify and clearly state the factual information extracted from the domain graph.
   - If the structured data directly answers the question, use it as the **primary basis** of your response.
   - Rephrase it into natural text (do not just list JSON keys or properties).
   - Start your answer with:
     `"According to the domain knowledge graph, ..."`

2. **Contextual Enrichment**
   - Use the vector article context **only to elaborate or explain** the domain facts.
   - Never contradict or replace domain graph data.
   - Attribute such context explicitly, e.g.,
     `"Supporting literature indicates..."` or
     `"According to recent studies..."`

3. **Attribution and Clarity**
   - Clearly distinguish between information from the **Domain Graph** and from **Vector Articles**.
   - If sources disagree, favor the Domain Graph but mention the discrepancy carefully.

4. **Response Structure**
   - Paragraph 1: Factual answer (domain graph)
   - Paragraph 2: Supporting evidence (vector articles)
   - Paragraph 3 (optional): Summary or implications

5. **Style**
   - Write as a scientific expert, neutral and precise.
   - Avoid speculation or general statements not supported by the data.

---

Answer:
"""
        )

        # Build RAG chain using LCEL
        rag_chain = (
            {
                "domain_context": RunnableLambda(
                    lambda x: self._format_context(
                        self.domain_retriever.get_relevant_documents(x.get("question", "")),
                        "Domain Graph"
                    )
                ),
                "vector_context": RunnableLambda(
                    lambda x: self._format_context(
                        self.vector_retriever.get_relevant_documents(x.get("question", "")),
                        "Vector Search"
                    )
                ),
                "question": RunnablePassthrough()
            }
            | prompt
            | self.llm
        )

        return rag_chain

    def query(self, question: str) -> HybridRAGResult:
        """Execute the RAG pipeline"""
        print(f"\n{'='*80}")
        print(f"QUERY: {question}")
        print(f"{'='*80}\n")

        start_time = datetime.now()

        # Build chain
        rag_chain = self.build_rag_chain()

        # Invoke chain with proper input
        answer = rag_chain.invoke({"question": question})

        # Get individual contexts for metadata
        domain_docs, vector_docs = self._parallel_retrieve(question)
        domain_context = self._format_context(domain_docs, "Domain Graph")
        vector_context = self._format_context(vector_docs, "Vector Search")

        execution_time = (datetime.now() - start_time).total_seconds()

        return HybridRAGResult(
            answer=answer,
            domain_graph_context=domain_context,
            vector_context=vector_context,
            sources_used={
                "domain_graph_results": len(domain_docs),
                "vector_search_results": len(vector_docs),
                "execution_time": execution_time,
                "llm_model": self.llm_model
            }
        )

    def close(self):
        """Close all connections"""
        self.domain_retriever.close()
        self.vector_retriever.close()


In [ ]:
# ============================================================================
# MAIN USAGE
# ============================================================================

def main():
    """Main execution"""

    print("\n" + "="*80)
    print("HYBRID RAG PIPELINE - FIXED VERSION")
    print("="*80)
    print("\nBefore running, make sure:")
    print("1. Ollama is running: ollama serve")
    print("2. Model is pulled: ollama pull mistral")
    print("3. Neo4j is configured with proper credentials")
    print("="*80 + "\n")

    try:

        pipeline = HybridRAGPipeline(
            neo4j_uri="neo4j+s://0568253a.databases.neo4j.io",
            neo4j_user="neo4j",
            neo4j_password="sR3IyAZfLN4p5nhMlsR7_bX99816dSDATZ2B4guGCV0",
            llm_model="mistral",
            embedding_model_name="all-MiniLM-L6-v2",
            ollama_base_url="http://localhost:11434"
        )

        # Test query
        question = "What are the genetic markers used for identification Hyalomma marginatum ?"
        result = pipeline.query(question)

        print("\n" + "-"*80)
        print("DOMAIN GRAPH CONTEXT:")
        print("-"*80)
        print(result.domain_graph_context)

        print("\n" + "-"*80)
        print("VECTOR SEARCH CONTEXT:")
        print("-"*80)
        print(result.vector_context)

        print("\n" + "-"*80)
        print("FINAL ANSWER:")
        print("-"*80)
        print(result.answer)

        print("\n" + "-"*80)
        print("METADATA:")
        print("-"*80)
        print(json.dumps(result.sources_used, indent=2))

        pipeline.close()

    except Exception as e:
        print(f"\n[ERROR] {e}")
        print("\nTroubleshooting:")
        print("✓ Is Ollama running? (ollama serve)")
        print("✓ Is Neo4j running and credentials correct?")
        print("✓ Are all dependencies installed?")


if __name__ == "__main__":
    main()


HYBRID RAG PIPELINE - FIXED VERSION

Before running, make sure:
1. Ollama is running: ollama serve
2. Model is pulled: ollama pull mistral
3. Neo4j is configured with proper credentials

[Init] Loading OllamaLLM (using langchain_ollama)...
[Init] Loading Sentence Transformer embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Init] Initializing Domain Graph Retriever...
[Init] Initializing Vector Graph Retriever...
[Init] Pipeline ready!


QUERY: What are the genetic markers used for identification Hyalomma marginatum ?

[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})
RETURN s.genetic_markers...
[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})
RETURN s.genetic_markers...

--------------------------------------------------------------------------------
DOMAIN GRAPH CONTEXT:
--------------------------------------------------------------------------------
1. {
  "s.genetic_markers": [
    "12S rRNA",
    "16S"
  ]
}

--------------------------------------------------------------------------------
VECTOR SEARCH CONTEXT:
--------------------------------------------------------------------------------
1. [Genetic diversity of H. marginatum in T

In [ ]:

# 1. Installer les dépendances manquantes
!pip install fastapi uvicorn pyngrok nest_asyncio -q

# 2. Importer ce qui manque
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# 3. Créer l'API FastAPI
app = FastAPI(title="RAG API")

# 4. Définir le format de la requête
class QueryRequest(BaseModel):
    query: str
    context_window: int = 3
    temperature: float = 0.7
    max_tokens: int = 512

# 5. Définir le format de la réponse
class QueryResponse(BaseModel):
    answer: str
    sources: list
    execution_time: float

# 6. Variable globale pour votre pipeline RAG
# (celui que vous avez déjà créé avec HybridRAGPipeline)
rag_pipeline = None

# 7. Initialiser le pipeline au démarrage
@app.on_event("startup")
async def startup():
    global rag_pipeline
    print("🚀 Initialisation du pipeline RAG...")

    # Utiliser VOTRE pipeline existant
    rag_pipeline = HybridRAGPipeline(
        neo4j_uri="neo4j+s://0568253a.databases.neo4j.io",
        neo4j_user="neo4j",
        neo4j_password="sR3IyAZfLN4p5nhMlsR7_bX99816dSDATZ2B4guGCV0",
        llm_model="mistral",
        embedding_model_name="all-MiniLM-L6-v2",
        ollama_base_url="http://localhost:11434"
    )
    print("Pipeline RAG prêt!")

# 8. Endpoint principal pour les questions
@app.post("/query", response_model=QueryResponse)
async def query_endpoint(request: QueryRequest):
    """Endpoint pour poser des questions au RAG"""
    try:
        # Appeler VOTRE fonction query existante
        result = rag_pipeline.query(request.query)

        # Formater la réponse
        return QueryResponse(
            answer=result.answer,
            sources=[
                {"type": "domain_graph", "content": result.domain_graph_context},
                {"type": "vector_search", "content": result.vector_context}
            ],
            execution_time=result.sources_used.get("execution_time", 0)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 9. Endpoint de santé
@app.get("/health")
async def health():
    return {"status": "healthy", "model": "mistral"}

# 10. Fonction pour démarrer le serveur avec ngrok
def start_server(ngrok_token: str):
    """
    Démarre le serveur avec ngrok

    Args:
        ngrok_token: Votre token ngrok (à obtenir sur https://ngrok.com)
    """
    print("\n" + "="*80)
    print("DÉMARRAGE DU SERVEUR RAG")
    print("="*80 + "\n")

    # Configurer ngrok
    ngrok.set_auth_token(ngrok_token)


    public_url = ngrok.connect(8000)
    print(f"\n URL PUBLIQUE: {public_url}")
    print("\n COPIEZ CETTE URL - vous en aurez besoin pour votre PC local!\n")
    print("="*80 + "\n")


    uvicorn.run(app, host="0.0.0.0", port=8000)


/tmp/ipython-input-2285966706.py:38: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [ ]:


import nest_asyncio
import asyncio

nest_asyncio.apply()




In [ ]:
!pkill ngrok


In [ ]:
import threading, time

threading.Thread(target=start_server, kwargs={
    "ngrok_token": "347SE8m64mzndxGq04SEJJO9qZA_4rCSSo62eDqjGHrRmL1q9"
}).start()

while True:
    time.sleep(1)



DÉMARRAGE DU SERVEUR RAG


🌐 URL PUBLIQUE: NgrokTunnel: "https://minimus-sickeningly-baylee.ngrok-free.dev" -> "http://localhost:8000"

⚠️  COPIEZ CETTE URL - vous en aurez besoin pour votre PC local!




INFO:     Started server process [947]
INFO:     Waiting for application startup.


🚀 Initialisation du pipeline RAG...
[Init] Loading OllamaLLM (using langchain_ollama)...
[Init] Loading Sentence Transformer embeddings...


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[Init] Initializing Domain Graph Retriever...
[Init] Initializing Vector Graph Retriever...
[Init] Pipeline ready!

✅ Pipeline RAG prêt!
INFO:     102.157.95.10:0 - "GET /health HTTP/1.1" 200 OK

QUERY: What are the genetic markers for Hyalomma marginatum?

[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})
RETURN s.genetic_markers...
[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})
RETURN s.genetic_markers...
INFO:     102.157.95.10:0 - "POST /query HTTP/1.1" 200 OK

QUERY: Where is Hyalomma marginatum found?

[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})-[:FOUND_IN]->(l:Location)
RETURN...
[Vector Search] Generated embedding of dimension 384
[Domain Graph] Generated Cypher: MATCH (s:Species {name: 'Hyalomma marginatum'})-[:FOUND_IN]

KeyboardInterrupt: 